In [1]:
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import os
import pickle
from torchvision import transforms
import pytorch_lightning as pl
import torch

# load own code
import sys
sys.path.append('../')
from sleeplib.Resnet_15.model import FineTuning
from sleeplib.datasets import BonoboDataset, ContinousToSnippetDataset
# this holds all the configuration parameters
from sleeplib.config import Config
import pickle
from tqdm import tqdm
from pytorch_lightning.callbacks import ModelCheckpoint
from torch.utils.data import DataLoader
from torchvision import transforms

from sleeplib.datasets import BonoboDataset , ContinousToSnippetDataset, NEWBonoboDataset
from sleeplib.montages import CDAC_bipolar_montage,CDAC_common_average_montage,CDAC_combine_montage,con_combine_montage,CDAC_noECG_montage
from sleeplib.transforms import cut_and_jitter, channel_flip


In [2]:
path_model = 'Models/YOUR_MODEL_NAME'
# load config file
with open(path_model+'/config.pkl', 'rb') as f:
   config = pickle.load(f)
# load dataset
df = pd.read_csv('/data/0shared/lijun/code/Spike_detection/localization.csv',sep=',')
# fraction filter
mode_filter = df['Mode'] == 'Test'

test_df = df[mode_filter]
print(f'there are {len(test_df)} test samples')

there are 1268 test samples


In [3]:
# set up dataloader to predict all samples in test dataset
transform_val = transforms.Compose([cut_and_jitter(windowsize=config.WINDOWSIZE,max_offset=0,Fq=config.FQ)])
combine_montage = CDAC_combine_montage()
noECG_montage = CDAC_noECG_montage()

test_dataset = NEWBonoboDataset(test_df, '/data/0shared/lijun/data/bonobo/npy_1', 
                           transform=transform_val,
                           montage = combine_montage,
                           window_size = config.WINDOWSIZE
                          )
test_dataloader = DataLoader(test_dataset, batch_size=32,shuffle=False,num_workers=os.cpu_count())


'''
Bonobo_con = ContinousToSnippetDataset('/home/ubuntu/data/Bonobo01742_0.mat',montage=con_combine_montage)
con_dataloader = DataLoader(Bonobo_con, batch_size=config.BATCH_SIZE,shuffle=False,num_workers=os.cpu_count())

for x, y in con_dataloader:
    with torch.no_grad():
        print(x.shape)
        print(y)
        break
'''

"\nBonobo_con = ContinousToSnippetDataset('/home/ubuntu/data/Bonobo01742_0.mat',montage=con_combine_montage)\ncon_dataloader = DataLoader(Bonobo_con, batch_size=config.BATCH_SIZE,shuffle=False,num_workers=os.cpu_count())\n\nfor x, y in con_dataloader:\n    with torch.no_grad():\n        print(x.shape)\n        print(y)\n        break\n"

In [4]:
# load pretrained model
model = FineTuning.load_from_checkpoint('/data/0shared/lijun/code/Spike_detection/Models/YOUR_MODEL_NAME/spike_det_weights-v11.ckpt',
                                        lr=config.LR,
                                        head_dropout=config.HEAD_DROPOUT,
                                        n_channels=config.N_CHANNELS,
                                        n_fft=config.N_FFT,
                                        hop_length=config.HOP_LENGTH,
                                       )
                                        #map_location=torch.device('cpu') add this if running on CPU machine
# init trainer
trainer = pl.Trainer(devices=1, accelerator="gpu",fast_dev_run=False,enable_progress_bar=False)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [5]:
# predict all samples
#['Fp1', 'F3', 'C3', 'P3', 'F7', 'T3', 'T5', 'O1', 'Fz', 'Cz', 'Pz', 'Fp2', 'F4', 'C4', 'P4', 'F8', 'T4', 'T6', 'O2']
result = trainer.predict(model,test_dataloader)
preds = [] # Empty array
reg = []   # Empty array

for i in tqdm(range(len(result))):
    preds = [item[0] for item in result]
    reg = [item[1] for item in result]


preds = np.concatenate(preds)
reg = np.concatenate(reg)

You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
100%|██████████| 40/40 [00:00<00:00, 109369.07it/s]


In [6]:
# store results
results = test_df[['event_file','fraction_of_yes', 'Mode', 'location']].copy()
results['preds'] = preds
channels = ['xFp1', 'xF3', 'xC3', 'xP3', 'xF7', 'xT3', 'xT5', 'xO1', 'xFz', 'xCz', 'xPz', 'xFp2', 'xF4', 'xC4', 'xP4', 'xF8', 'xT4', 'xT6', 'xO2']

for i, channel in enumerate(channels):
    results[channel] = reg[:,i]

results.to_csv(path_model+'/predictions.csv',index=False)